In [3]:
!pip install requests python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.3 MB/s eta 0:00:00


In [5]:
import requests
from docx import Document
import os

In [ ]:
subscription_key = "BpOxZyHafAxUt7VjZ46Hs6OwAY6X5mrO4bxfMbfYvswSYvovsX28JQQJ99CBACYeBjFXJ3w3AAAbACOGCpdA" 
endpoint = "https://api.cognitive.microsofttranslator.com"
location = "eastus"
language_destination = 'pt-br'

def translator_text(text, target_language):
    path = '/translate'
    constructed_url = endpoint + path
    headers = {
        'Ocp-Apim-Subscription-Key': subscription_key,
        'Ocp-Apim-Subscription-Region': location,
        'Content-type': 'application/json',
        'X-ClientTraceId': str(os.urandom(16))
    }

    body = [{
        'text' : text

    }]
    params = {
        'api-version': '3.0',
        'from': 'en',
        'to': target_language
    }

    api_response = requests.post(constructed_url, params=params, headers=headers, json=body)

    if api_response.status_code == 200:
        response_json = api_response.json()
        if response_json and len(response_json) > 0 and 'translations' in response_json[0] and len(response_json[0]['translations']) > 0:
            return response_json[0]['translations'][0]['text']
        else:
            print(f"Error: Unexpected API response format: {response_json}")
            return "Translation failed: Unexpected response format"
    else:
        print(f"Error: API request failed with status code {api_response.status_code}: {api_response.text}")
        return "Translation failed: API error"

In [28]:
translator_text("I know you're somewhere out there, somewhere far away", lagnage_destination )

'Eu sei que você está em algum lugar lá fora, em algum lugar bem longe'

In [41]:
def translate_document(path):
  document = Document(path)
  full_text = []
  for paragraph in document.paragraphs:
    translated_text = translator_text(paragraph.text, language_destination)
    full_text.append(translated_text)

  translated_doc = Document()
  for line in full_text:
    translated_doc.add_paragraph(line)

  path_tranalated = path.replace(".docx", f"_{language_destination}.docx")
  translated_doc.save(path_tranalated)
  return path_tranalated

In [42]:
input_file = "/content/Musica_bruno_Mars.docx"
translate_document(input_file)

'/content/Musica_bruno_Mars_pt-br.docx'

In [48]:
!pip install requests beautifulsoup4 openai langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.1 MB/s eta 0:00:00


In [50]:
import requests
from bs4 import BeautifulSoup

In [67]:
def extract_text_from_url(url):
  response = requests.get(url)

  if response.status_code == 200:
      soup = BeautifulSoup(response.text, 'html.parser')
      for script_or_style in soup(['script', 'style']):
          script_or_style.decompose()
      text = soup.get_text()
      # Limpar texto
      linhas = (line.strip() for line in text.splitlines())
      parts = (phrase.strip() for line in linhas for phrase in line.split("  "))
      text = '\n'.join(part for part in parts if part)
      return text
  else:
      print(f'Failed to fetch the URL. Status code: {response.status_code}')
      return None

In [68]:
extract_text_from_url('https://dev.to/kenakamu/azure-open-ai-in-vnet-3alo')

"Azure Open AI in VNet - DEV Community\nSkip to content\nNavigation menu\nSearch\nPowered by Algolia\nSearch\nLog in\nCreate account\nDEV Community\nClose\nAdd reaction\nLike\nUnicorn\nExploding Head\nRaised Hands\nFire\nJump to Comments\nSave\nBoost\nMore...\nCopy link\nCopy link\nCopied to Clipboard\nShare to X\nShare to LinkedIn\nShare to Facebook\nShare to Mastodon\nReport Abuse\nKenichiro Nakamura\nPosted on Oct 12, 2023\nAzure Open AI in VNet\n#azure\n#openai\n#security\nGPT models are hosted in multiple service vendor at the moment, and Microsoft Azure is one of them.\nEven though the models themselves are the same, there are many differences including:\ncost\nfunctionalities\ntype of models and versions\ngeo location\nsecurity\nsupport\netc.\nOne of the most important aspects when we use it in an Enterprise Environment is, of course, security.\nBy using Azure network security features with Azure Open AI, customers can consume the Open AI service from and within the VNet, theref

In [70]:
from langchain_openai.chat_models.azure import AzureChatOpenAI


In [95]:
from langchain_openai.chat_models.azure import AzureChatOpenAI

client = AzureChatOpenAI(
    azure_endpoint = "https://oai-dio-bootcamp-dev-eastus.openai.azure.com/",
    api_key = '6utJFrEKMDBW9wei1uZ7YPLej4vWmhb2WvhHOdf6jhewLU3Cp99eJQQJ99CBACYeBjFXJ3w3AAABACOGwtmi',
    api_version = '2024-07-18',
    #api_version = '2024-02-15-preview',
    deployment_name = "gpt-4o-mini",
    max_retries=0
)



In [96]:
def translate_article(text, lang):
  messages = [
      ("system" , "Você atua como tradutor de textos"),
      ("user", f"Traduza o {text} para o idioma {lang} e responda me markdown")
  ]
  response = client.invoke(messages)
  print(response.content)
  return response.content

In [97]:
url = 'https://dev.to/kenakamu/azure-open-ai-in-vnet-3alo'
text = extract_text_from_url(url)
article = translate_article(text, "pt-br")
print(article)

NotFoundError: Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}